# Greedy evaluation

In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib widget

In [2]:
import ray
import numba as nb
import numpy as np
import xarray.ufuncs as xrf
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
import pickle
from pathlib import Path
from itertools import product

from tqdm.auto import trange, tqdm
from math import radians, pi, sin
from numba.experimental import jitclass
from IPython.display import display, JSON
import ipywidgets as widgets

from ray import tune
from ray.tune.registry import register_env
from ray.rllib.agents import ppo
from ray.rllib.models import ModelCatalog
from ray.rllib.models.tf.tf_modelv2 import TFModelV2
from ray.rllib.models.tf.fcnet import FullyConnectedNetwork
from ray.tune.utils.log import Verbosity
from ray.tune import JupyterNotebookReporter
from ray.rllib.models.preprocessors import NoPreprocessor
from ray.rllib.utils.filter import NoFilter
# Verbosity.V0_MINIMAL
# Verbosity.V1_EXPERIMENT
# Verbosity.V2_TRIAL_NORM
# Verbosity.V3_TRIAL_DETAILS

from cw.filters import smooth_signal
from cw.vdom import hyr
from traj1.logger import Logger
from environment import LauncherV1SubOrbital, Stage, AP_PITCH_RATE_CONTROL, wrap_angle
from sufcnet import SUFullyConnectedNetwork

Instructions for updating:
non-resource variables are not supported in the long term


## Initialize

In [3]:
register_env("LauncherV1SubOrbital", LauncherV1SubOrbital)
ModelCatalog.register_custom_model("SUFullyConnectedNetwork", SUFullyConnectedNetwork)

In [4]:
hyr(ray.init(address="auto", dashboard_host="0.0.0.0"))

2021-05-20 08:13:12,940	INFO worker.py:640 -- Connecting to existing Ray cluster at address: 10.10.3.2:6379


dict[9]:

node_ip_addressstr[9]:

10.10.3.2

raylet_ip_addressstr[9]:

10.10.3.2

redis_addressstr[14]:

10.10.3.2:6379

object_store_addressstr[67]:

/tmp/ray/session_2021-05-20_08-07-04_856591_57/sockets/plasma_store

raylet_socket_namestr[61]:

/tmp/ray/session_2021-05-20_08-07-04_856591_57/sockets/raylet

webui_urlstr[12]:

0.0.0.0:8265

session_dirstr[46]:

/tmp/ray/session_2021-05-20_08-07-04_856591_57

metrics_export_portint:

60679

node_idstr[56]:

4db3cc9c67cb36228a1610543ae73293a7a904358305320caca558bd

In [5]:
trainer = ppo.PPOTrainer(
    env="LauncherV1SubOrbital",
    config={
        "num_workers": 0,
        "model": {
            "custom_model": "SUFullyConnectedNetwork",
            "custom_model_config": {
#                 "dropouts": [0.2, 0.2],
                "training": False
            },
            
            "fcnet_hiddens": [32, 32],
            "fcnet_activation": "relu",
        },
    },
)

2021-05-20 08:13:13,667	INFO trainer.py:669 -- Tip: set framework=tfe or the --eager flag to enable TensorFlow eager execution
2021-05-20 08:13:13,670	INFO trainer.py:694 -- Current log_level is WARN. For more information, set 'log_level': 'INFO' / 'DEBUG' or use the -v and -vv flags.
2021-05-20 08:13:28,395	INFO trainable.py:101 -- Trainable.setup took 14.737 seconds. If your trainable is slow to initialize, consider setting reuse_actors=True to reduce actor creation overheads.


In [6]:
policy = trainer.get_policy()
policy.model.base_model.summary()

Model: "model"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
observations (InputLayer)       [(None, 3)]          0                                            
__________________________________________________________________________________________________
fc_1 (Dense)                    (None, 32)           128         observations[0][0]               
__________________________________________________________________________________________________
fc_value_1 (Dense)              (None, 32)           128         observations[0][0]               
__________________________________________________________________________________________________
fc_2 (Dense)                    (None, 32)           1056        fc_1[0][0]                       
______________________________________________________________________________________________

In [7]:
env = LauncherV1SubOrbital()

logger = Logger()
logger.register_time_attribute(env.sim, "t")
logger.register(env.sim, "env", [
    "h",
    "gamma_e", "theta_e", "theta_i_dot",
    "ap_comm_gamma_e", "ap_comm_theta_e",
    "action_autopilot_mode", "action_autopilot_reference", "vii", "xii", "fii_thrust", "mass", "mass_dot"])

## Load and evaluate

In [8]:
# run_path = Path(r"/home/jovyan/trajectory_optimization_1/work/max_altitude/results/ppo_basic/PPO_LauncherV1SubOrbital_8f4ab_00000_0_2021-05-15_13-00-44")
# run_path = Path(r"/home/jovyan/trajectory_optimization_1/work/max_altitude/results/ppo_basic/PPO_LauncherV1SubOrbital_93ecb_00000_0_2021-05-16_22-03-42")
# run_path = Path(r"/home/jovyan/trajectory_optimization_1/work/max_altitude/results/ppo_basic/PPO_LauncherV1SubOrbital_a152e_00000_0_2021-05-17_02-07-28")
# run_path = Path(r"/home/jovyan/trajectory_optimization_1/work/max_altitude/results/ppo_basic/PPO_LauncherV1SubOrbital_fb876_00000_0_2021-05-17_02-45-47")
# run_path = Path(r"/home/jovyan/trajectory_optimization_1/work/max_altitude/results/ppo_basic/PPO_LauncherV1SubOrbital_e522c_00000_0_2021-05-17_03-06-38")
# run_path = Path(r"/home/jovyan/trajectory_optimization_1/work/max_altitude/results/ppo_basic/PPO_LauncherV1SubOrbital_dcd00_00000_0_2021-05-17_03-56-30")  # 271
# run_path = Path(r"/home/jovyan/trajectory_optimization_1/work/max_altitude/results/ppo_basic/PPO_LauncherV1SubOrbital_49fa6_00000_0_2021-05-17_13-32-13")
run_path = Path(r"/home/jovyan/trajectory_optimization_1/work/max_altitude/results/ppo_basic/PPO_LauncherV1SubOrbital_60644_00000_0_2021-05-17_15-13-04")

In [9]:
def get_idxs(run_path):
    yield from sorted(int(checkpoint_path.name[-6:]) for checkpoint_path in run_path.glob("checkpoint_*"))


def checkpoint_path(run_path, idx=None):
    if idx is None:
        idx = max(get_idxs(run_path))
    checkpoint_path = run_path / f"checkpoint_{idx:06}/checkpoint-{idx}"
    if checkpoint_path.exists():
        return checkpoint_path


def restore_trainer(load_run_path=None, idx=None):
    load_run_path = load_run_path or run_path
    if path := checkpoint_path(load_run_path, idx):
        trainer.restore(str(path))
        print(f"Restored: {path}")
    else:
        print(f"WARNING: Checkpoint not found {path}")

In [45]:
restore_trainer(idx=2251)

2021-05-20 08:18:40,973	INFO trainable.py:377 -- Restored on 10.10.3.2 from checkpoint: /home/jovyan/trajectory_optimization_1/work/max_altitude/results/ppo_basic/PPO_LauncherV1SubOrbital_60644_00000_0_2021-05-17_15-13-04/checkpoint_002251/checkpoint-2251
2021-05-20 08:18:40,974	INFO trainable.py:385 -- Current state after restoring: {'_iteration': 2251, '_timesteps_total': None, '_time_total': 13826.433351516724, '_episodes_total': 27279}


Restored: /home/jovyan/trajectory_optimization_1/work/max_altitude/results/ppo_basic/PPO_LauncherV1SubOrbital_60644_00000_0_2021-05-17_15-13-04/checkpoint_002251/checkpoint-2251


In [46]:
env = LauncherV1SubOrbital({
    "initial_theta_e": radians(70),
    "theta_e_random_window": None,
})

logger = Logger()
logger.register_time_attribute(env.sim, "t")
logger.register(env.sim, "env", [
    "h", "reward",
    "gamma_e", "theta_e", "theta_i_dot",
    "ap_comm_gamma_e", "ap_comm_theta_e",
    "action_autopilot_mode", "action_autopilot_reference", "vii", "xii", "fii_thrust", "mass", "mass_dot"])

In [47]:
def simulate():
    episode_reward = 0
    done = False
    obs = env.reset()
    while not done:
        action = trainer.compute_action(obs, explore=False)
#         action = pc_policy(obs)
        obs, reward, done, info = env.step(action)
        episode_reward += reward
        logger.step()

    return logger.episode_finish()

In [48]:
episode_result = simulate()

In [49]:
# plt.figure()
f, (ax1, ax2, ax3) = plt.subplots(3, 1)
episode_result.env_gamma_e.plot.line(x="t", ax=ax1)
ax1.axhline(1.5)
smooth_signal(episode_result.env_action_autopilot_reference, wn=0.99).plot.line(x="t", ax=ax2)
episode_result.env_h.plot.line(x="t", ax=ax3)
plt.tight_layout()

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [25]:
episode_result

<xarray.Dataset>
Dimensions:                         (d_2_0: 2, t: 464)
Coordinates:
  * t                               (t) float64 0.05 0.1 0.15 ... 23.15 23.2
Dimensions without coordinates: d_2_0
Data variables: (12/14)
    env_h                           (t) float64 1.0 0.9959 ... 700.0 703.2
    env_reward                      (t) float64 0.0 0.0 0.0 ... 0.0 1.312e-08
    env_gamma_e                     (t) float64 -1.571 0.1868 ... 0.5619 0.5605
    env_theta_e                     (t) float64 0.7854 0.7854 ... 0.7861 0.7861
    env_theta_i_dot                 (t) float64 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0
    env_ap_comm_gamma_e             (t) float64 nan nan nan nan ... nan nan nan
    ...                              ...
    env_action_autopilot_reference  (t) float64 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0
    env_vii                         (t, d_2_0) float64 -4.975e-18 ... 63.26
    env_xii                         (t, d_2_0) float64 1.064e-10 ... 1.738e+06
    env_fii_thrust                  (t, d_2_0) float64 4.808 4.808 ... 0.0 0.0
    env_mass                        (t) float64 1.2 1.2 1.199 ... 1.0 0.9998 1.0
    env_mass_dot                    (t) float64 -0.008665 -0.008665 ... 0.0 0.0

## Action plot

In [50]:
def actions(alts=None, resolution=100):
    thetas = np.linspace(-pi, pi, resolution)
    gammas = np.linspace(-pi, pi, resolution)
    alts = alts or [0]
    
    policy = trainer.get_policy()
    assert isinstance(trainer.workers.local_worker().preprocessors['default_policy'], NoPreprocessor)
    assert isinstance(trainer.workers.local_worker().filters['default_policy'], NoFilter)

    observations = list(product(gammas, thetas, alts))
    return policy.compute_actions(observations, explore=False)[0].reshape((resolution, resolution, len(alts)))

In [51]:
restore_trainer(idx=2251)

2021-05-20 08:19:17,217	INFO trainable.py:377 -- Restored on 10.10.3.2 from checkpoint: /home/jovyan/trajectory_optimization_1/work/max_altitude/results/ppo_basic/PPO_LauncherV1SubOrbital_60644_00000_0_2021-05-17_15-13-04/checkpoint_002251/checkpoint-2251
2021-05-20 08:19:17,218	INFO trainable.py:385 -- Current state after restoring: {'_iteration': 2251, '_timesteps_total': None, '_time_total': 13826.433351516724, '_episodes_total': 27279}


Restored: /home/jovyan/trajectory_optimization_1/work/max_altitude/results/ppo_basic/PPO_LauncherV1SubOrbital_60644_00000_0_2021-05-17_15-13-04/checkpoint_002251/checkpoint-2251


In [52]:
actions_ = actions()

In [56]:
fig = plt.figure()
ax = plt.gca()
vscale = min(max(actions_.max(), -actions_.min()), 1)
actions_image = plt.imshow(actions_[::-1,...], extent=[-pi, pi, -pi, pi], cmap="BrBG", vmin=-vscale, vmax=vscale)
ax.axvline(0.5 * pi)
ax.axhline(0.5 * pi)
# plt.plot(0.5 * pi, 0.5 * pi, "x")
plt.colorbar()

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

## Action animation

In [54]:
def init():
    return actions_image,


def update(frame):
    ax.set_title(frame)
    actions_image.set_data(actions([frame])[::-1,...])
    return actions_image,


frames = np.linspace(0, 1, 10)
ani = FuncAnimation(fig, update, frames=frames, init_func=init, blit=True, interval=100)

In [55]:
plt.show()

## P-controller policy

In [47]:
def pc_policy(obs, k_gamma=10, k_theta=10):
    return k_gamma * (0.5 * pi - obs[0])

In [29]:
def pc_actions(resolution=100):
    thetas = np.linspace(-pi, pi, resolution)
    gammas = np.linspace(-pi, pi, resolution)
    actions = np.empty((resolution, resolution))
    for (gamma_i, gamma), (theta_i, theta) in product(enumerate(thetas), enumerate(gammas)):
        actions[gamma_i, theta_i] = trainer.compute_action([gamma, theta, 0.], explore=False)
#         actions[gamma_i, theta_i] = gamma
    return actions

In [30]:
actions_ = pc_actions()
fig = plt.figure()
vscale = max(actions_.max(), -actions_.min())
plt.imshow(actions_[::-1,...], extent=[-pi, pi, -pi, pi], cmap="BrBG", vmin=-vscale, vmax=vscale)
plt.plot(0.5 * pi, 0.5 * pi, "x")
plt.colorbar()

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [78]:
def foo(i):
    s = 1
    n = 5
    return (i / (n-1) - 0.5) * s * 2

In [80]:
plt.figure()
plt.plot(list(map(foo, [0, 1, 2, 3, 4])), "x")


<ipython-input-80-4a9393706554>:1: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`).
  plt.figure()


Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [33]:
0.99**450

0.010860193639877882